In [ ]:
import os
import sys
os.environ['JULIA_DEPOT_PATH'] = '/pscratch/sd/s/sferrett/.julia'
os.environ['PYTHON_JULIAPKG_PROJECT'] = '/pscratch/sd/s/sferrett/.julia/environments/pyjuliapkg'
sys.path.insert(0,os.path.abspath('..'))
import gc
import json
import time
import pickle
import shutil
import tempfile
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import xarray as xr
import pandas as pd
import proplot as pplt
from pysr import PySRRegressor, jl
from scripts.data.classes import PredictionWriter
jl.seval('safe_pow(x, y) = abs(x)^y')
pplt.rc.update({'figure.dpi':100})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)

SPLITSDIR    = CONFIGS['filepaths']['splits']
PREDSDIR     = CONFIGS['filepaths']['predictions']
WEIGHTSDIR   = CONFIGS['filepaths']['weights']
MODELSDIR    = CONFIGS['filepaths']['models']
TARGETVAR    = CONFIGS['variables']['target']
SRRUNS       = CONFIGS['experiments']['sr']['runs']
RUNNAME      = list(SRRUNS.keys())[0]
RUNCONFIG    = SRRUNS[RUNNAME]
FIELDVARS    = RUNCONFIG['fieldvars']
LOCALVARS    = RUNCONFIG['localvars']
FEATURESFROM = RUNCONFIG.get('features_from',None)
PREDICTORS   = FIELDVARS + LOCALVARS
OUTDIR       = os.path.join(MODELSDIR,'sr',RUNNAME)
SUBSETSIZE   = 5000
SEED         = 42
SEARCHMODE   = 'test'

print(f'Run: {RUNNAME}')
if FEATURESFROM:
    print(f'  Field vars (kernel-integrated from {FEATURESFROM} weights): {FIELDVARS}')
else:
    print(f'  Field vars (from normalized splits): {FIELDVARS}')
print(f'  Local vars (from normalized splits): {LOCALVARS}')
print(f'  Target: {TARGETVAR}')

In [ ]:
def kernel_integrate(fields,weights,dlev,mask=None):
    weighted = fields * weights[None,:,:] * dlev[None,None,:]
    if mask is not None:
        weighted = weighted * mask[:,None,:]
    return weighted.sum(axis=2)

def load_split(splitname,fieldvars,localvars,targetvar,splitsdir,
               weightsdir=None,featuresfrom=None,seed=42):
    splitpath = os.path.join(splitsdir,f'norm_{splitname}.h5')
    splitds   = xr.open_dataset(splitpath,engine='h5netcdf')
    refda = splitds[targetvar].transpose('time','lat','lon')
    ntime = splitds.sizes['time']
    columns = {}
    if featuresfrom:
        wpath   = os.path.join(weightsdir,f'{featuresfrom}_{seed}_weights.nc')
        wds     = xr.open_dataset(wpath,engine='h5netcdf')
        weights = wds['k1'].isel(seed=0).values
        wds.close()
        nlevs = splitds.sizes['lev']
        dlev  = splitds['dlev'].values
        fieldarrays = []
        for var in fieldvars:
            da = splitds[var].transpose('time','lat','lon','lev')
            fieldarrays.append(da.values.reshape(-1,nlevs))
        fields3d = np.stack(fieldarrays,axis=1)
        mask = splitds['surfmask'].transpose('time','lat','lon','lev').values.reshape(-1,nlevs)
        feats = kernel_integrate(fields3d,weights,dlev,mask)
        for i,var in enumerate(fieldvars):
            columns[var] = feats[:,i]
    else:
        for var in fieldvars:
            da = splitds[var]
            if 'time' in da.dims:
                arr = da.transpose('time','lat','lon').values
            else:
                arr = np.tile(da.values,(ntime,1,1))
            columns[var] = arr.ravel()
    for var in localvars:
        da = splitds[var]
        if 'time' in da.dims:
            arr = da.transpose('time','lat','lon').values
        else:
            arr = np.tile(da.values,(ntime,1,1))
        columns[var] = arr.ravel()
    X = pd.DataFrame(columns)
    y = refda.values.ravel()
    validmask = np.isfinite(X).all(axis=1).values & np.isfinite(y)
    splitds.close()
    return X,y,refda,validmask

In [ ]:
scalarcols = {}
targets = []
profilerows = []
maskrows = []
dlevvals = None

for splitname in ('train','valid'):
    splitpath = os.path.join(SPLITSDIR,f'norm_{splitname}.h5')
    splitds = xr.open_dataset(splitpath,engine='h5netcdf')
    ntime = splitds.sizes['time']
    targets.append(splitds[TARGETVAR].transpose('time','lat','lon').values.ravel())
    for var in LOCALVARS:
        da = splitds[var]
        if 'time' in da.dims:
            arr = da.transpose('time','lat','lon').values
        else:
            arr = np.tile(da.values,(ntime,1,1))
        scalarcols.setdefault(var,[]).append(arr.ravel())
    if FEATURESFROM:
        nlevs = splitds.sizes['lev']
        dlevvals = splitds['dlev'].values
        fieldarrays = []
        for var in FIELDVARS:
            da = splitds[var].transpose('time','lat','lon','lev')
            fieldarrays.append(da.values.reshape(-1,nlevs))
        profilerows.append(np.stack(fieldarrays,axis=1))
        maskrows.append(splitds['surfmask'].transpose('time','lat','lon','lev').values.reshape(-1,nlevs))
    else:
        for var in FIELDVARS:
            da = splitds[var]
            if 'time' in da.dims:
                arr = da.transpose('time','lat','lon').values
            else:
                arr = np.tile(da.values,(ntime,1,1))
            scalarcols.setdefault(var,[]).append(arr.ravel())
    splitds.close()

yfit = np.concatenate(targets)
fitcols = {k:np.concatenate(v) for k,v in scalarcols.items()}
fitdf = pd.DataFrame(fitcols)
fitdf['__target__'] = yfit
fitdf = fitdf.replace([np.inf,-np.inf],np.nan).dropna()
keepidx = fitdf.index.values
yfit = fitdf.pop('__target__').values

np.random.seed(SEED)
if len(yfit) > SUBSETSIZE:
    subidx = np.random.choice(len(yfit),size=SUBSETSIZE,replace=False)
    print(f'Subsampled {SUBSETSIZE} from {len(yfit)} clean train+valid points')
else:
    subidx = np.arange(len(yfit))
    print(f'Using all {len(yfit)} clean train+valid points (< {SUBSETSIZE})')

Xsub = fitdf.iloc[subidx].reset_index(drop=True)
ysub = yfit[subidx]

if FEATURESFROM:
    wpath = os.path.join(WEIGHTSDIR,f'{FEATURESFROM}_{SEED}_weights.nc')
    wds = xr.open_dataset(wpath,engine='h5netcdf')
    weights = wds['k1'].isel(seed=0).values
    wds.close()
    allprofiles = np.concatenate(profilerows,axis=0)
    allmasks = np.concatenate(maskrows,axis=0)
    subprofiles = allprofiles[keepidx[subidx]]
    submasks = allmasks[keepidx[subidx]]
    feats = kernel_integrate(subprofiles,weights,dlevvals,submasks)
    for i,var in enumerate(FIELDVARS):
        Xsub.insert(i,var,feats[:,i])
    del allprofiles,allmasks,subprofiles,submasks,profilerows,maskrows

print(f'Xsub: {Xsub.shape}, ysub: {ysub.shape}')
print(f'\nXsub summary:\n{Xsub.describe()}')
print(f'\nysub: mean={ysub.mean():.6f}, std={ysub.std():.6f}, min={ysub.min():.6f}, max={ysub.max():.6f}')

In [ ]:
nvars = len(PREDICTORS)
fig,axes = pplt.subplots(ncols=nvars,refwidth=3.5,refheight=3)
for i,var in enumerate(PREDICTORS):
    ax = axes[i]
    ax.scatter(Xsub[var],ysub,s=1,alpha=0.3)
    ax.format(xlabel=var,ylabel=TARGETVAR if i==0 else '')
fig.suptitle(f'{TARGETVAR} vs Predictors (n={len(ysub)})')
pplt.show()

In [ ]:
opcomplexity = {'+':1,'-':1,'*':1,'/':3,'safe_pow':3,'exp':4,'log':4}

searchparams = {
    'test':dict(niterations=5,populations=5,population_size=33,maxsize=15,timeout_in_seconds=300),
    'medium':dict(niterations=100,populations=20,population_size=50,maxsize=30,timeout_in_seconds=1800),
    'full':dict(niterations=10_000_000,populations=30,population_size=100,maxsize=50,timeout_in_seconds=int(60*60*7.5)),
}

params = searchparams[SEARCHMODE]
print(f'Search mode: {SEARCHMODE}')
print(f'Parameters: {params}')

In [ ]:
os.makedirs(OUTDIR,exist_ok=True)
TMPDIR = tempfile.mkdtemp(prefix='pysr_')

model = PySRRegressor(
    niterations=params['niterations'],
    populations=params['populations'],
    population_size=params['population_size'],
    binary_operators=['+','-','*','/','safe_pow'],
    unary_operators=['exp','log'],
    complexity_of_operators=opcomplexity,
    complexity_of_variables=2,
    complexity_of_constants=1,
    maxsize=params['maxsize'],
    maxdepth=4,
    constraints={'safe_pow':(-1,1)},
    nested_constraints={
        'exp':{'exp':0,'log':0,'safe_pow':0},
        'safe_pow':{'safe_pow':0},
        'log':{'log':0,'exp':0}},
    extra_sympy_mappings={'safe_pow':lambda x,y:x**y},
    loss='loss(x, y) = (x - y)^2',
    model_selection='best',
    batching=len(ysub)>2000,
    batch_size=min(2000,len(ysub)),
    random_state=SEED,
    deterministic=True,
    multithreading=False,
    procs=0,
    tempdir=TMPDIR,
    temp_equation_file=True,
    delete_tempfiles=True,
    timeout_in_seconds=params['timeout_in_seconds'],
    progress=True)

t0 = time.time()
model.fit(Xsub.values,ysub,variable_names=PREDICTORS)
elapsed = time.time()-t0

shutil.rmtree(TMPDIR,ignore_errors=True)
print(f'\nSearch completed in {elapsed/60:.1f} minutes')
print(model)

In [ ]:
equations = model.equations_
print(equations[['equation','complexity','loss','score']].to_string())

In [ ]:
besteq = model.get_best()
msesub = np.mean((ysub-model.predict(Xsub.values))**2)

Xtest,ytest,refdatest,validtest = load_split('test',FIELDVARS,LOCALVARS,TARGETVAR,SPLITSDIR,WEIGHTSDIR,FEATURESFROM,SEED)
Xtest = Xtest[validtest].reset_index(drop=True)
ytest = ytest[validtest]
ypredtest = model.predict(Xtest.values)
msetest = np.mean((ytest-ypredtest)**2)

print(f'Best equation: {besteq["equation"]}')
print(f'  Complexity:   {besteq["complexity"]}')
print(f'  MSE fit subset: {msesub:.6f}')
print(f'  MSE test:       {msetest:.6f}')

In [ ]:
fig,ax = pplt.subplots(refwidth=5,refheight=3.5)
ax.scatter(equations['complexity'],equations['loss'],zorder=5)
ax.format(xlim=(0,equations['complexity'].max()+3),
          ylim=(equations['loss'].min()*0.9,equations['loss'].max()*1.1),
          xlabel='Complexity',ylabel='MSE Loss',
          title='Pareto Frontier of Discovered Equations')
for i,row in equations.iterrows():
    label = str(row['equation'])[:50]
    ax.annotate(label,xy=(row['complexity'],row['loss']),xytext=(5,5),textcoords='offset points',fontsize=5,clip_on=True)
pplt.show()

In [ ]:
modelpath = os.path.join(OUTDIR,f'{RUNNAME}.pkl')
with open(modelpath,'wb') as f:
    pickle.dump(model,f)
equations.to_csv(os.path.join(OUTDIR,'equations.csv'),index=False)

writer = PredictionWriter(SPLITSDIR,targetvar=TARGETVAR)
predarr = writer.predictions_to_array(ypredtest,validtest,refdatest)
predds  = writer.predictions_to_dataset(predarr[...,np.newaxis],refdatest)
writer.save(RUNNAME,predds,'predictions','test',PREDSDIR)

print(f'Model saved to: {modelpath}')
print(f'Equations saved to: {os.path.join(OUTDIR,"equations.csv")}')
print(f'Test predictions saved to: {PREDSDIR}')
gc.collect()

In [2]:
# FILEDIR = '/global/cfs/cdirs/m4334/sferrett/monsoon-discovery/data/interim'
# x  = xr.open_dataarray(f'{FILEDIR}/cape.nc').isel(time=slice(0,5),lat=slice(0,5),lon=slice(0,5)).values.ravel()
# y  = xr.open_dataarray(f'{FILEDIR}/bl.nc').isel(time=slice(0,5),lat=slice(0,5),lon=slice(0,5)).values.ravel()
# df = pd.DataFrame({'x':x,'y':y})

# df = df.replace([np.inf,-np.inf],np.nan).dropna()
# print(f'Data shape after cleaning: {df.shape}')
# print(df.describe())

# fig,ax = pplt.subplots(nrows=1,ncols=1)
# ax.format(xlabel='$CAPE_L$ (K)',ylabel='$B_L$ (m/s$^2$)')
# ax.scatter(df['x'],df['y'],alpha=0.6)
# pplt.show()

# model = PySRRegressor(
#     niterations=3,
#     populations=3,
#     population_size=33,
#     tournament_selection_n=2,
#     binary_operators=['+','-','*','/','safe_pow'],
#     unary_operators=['exp','log'],
#     complexity_of_operators={'+':1,'*':1,'-':1,'/':3,'safe_pow':3,'exp':4,'log':4},
#     complexity_of_variables=2,
#     complexity_of_constants=1,
#     maxsize=10,
#     constraints={'safe_pow':(-1,1)},
#     nested_constraints={
#         'exp':{'exp':0,'log':0,'safe_pow':0},
#         'safe_pow':{'safe_pow':0},
#         'log':{'log': 0,'exp':0}},  
#     extra_sympy_mappings={'safe_pow':lambda x,y:x**y},
#     batching=False,
#     batch_size=100,
#     loss='loss(x, y) = (x - y)^2',
#     model_selection='best',
#     random_state=42,
#     deterministic=True,
#     multithreading=False,
#     procs=0,
#     timeout_in_seconds=300)

# model.fit(df[['x']].values, df['y'].values)
# print(model)

# equations = model.equations_
# print(equations[['equation','complexity','loss','score']].head(10))

# best  = model.get_best()
# eqstr = best['sympy_format']
# ypred = best['lambda_format'](df[['x']].values)

# fig,ax = pplt.subplots(nrows=1,ncols=1,refheight=2,refwidth=2)
# ax.format(title='Pareto Frontier of Discovered Equations',xlabel='Complexity',ylabel='MSE Loss')
# ax.scatter(equations['complexity'],equations['loss'])
# for i,row in equations.iterrows():
#     ax.annotate(row['equation'],xy=(row['complexity'],row['loss']),xytext=(5,5),textcoords='offset points',fontsize=6)
# pplt.show()

# fig,ax = pplt.subplots(nrows=1,ncols=1,refwidth=2.5)
# ax.format(title=f'Best PySR Equation: {eqstr}',grid=True,xlabel='$y_{true}$',ylabel='$y_{pred}$')
# ax.plot(df['y'],df['y'],color='k',linestyle='--')
# ax.scatter(df['y'],ypred,alpha=0.6)
# pplt.show()